In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\start_vector
✅ Sys path: ['c:\\Users\\123\\Desktop\\start_vector', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [14]:
import logging
import pandas as pd
import aiohttp
import asyncio
from datetime import datetime, timedelta
import requests
import json
import time

from src_oop.core.my_gspread import GoogleTabs
from src_oop.core.utils_general import load_api_tokens, load_sima_land_tokens
from src_oop.jobs.sales_plan.config import annual_procurement_plan
from src_oop.core.database import Database

In [15]:
class SalesPlan:
    def __init__(self):
        # Таблица Годовой план закупа
        self._annual_procurement_plan_table = annual_procurement_plan.get("title")
        # Лист Поквартально в таблице
        self._annual_procurement_plan_sheet = annual_procurement_plan.get("quarter_sheet")
        self._conn_to_annual_procurement_plan_table = None

        # Подключение к БД
        self.engine = Database.get_engine()

    # ===
    # Подключение к гугл-таблицам
    # ===    

    # Подключение к таблице Годовой план закупа, листу Поквартально
    @property
    def google_conn_to_annual_procurement_plan_table(self):
        """Ленивое подключение к таблице Годовой план закупа, листу Поквартально"""
        if self._conn_to_annual_procurement_plan_table is None:
            self._conn_to_annual_procurement_plan_table = GoogleTabs(self._annual_procurement_plan_table, self._annual_procurement_plan_sheet)
        return self._conn_to_annual_procurement_plan_table
    
    # ===
    # Получение данных из гугл таблиц
    # ===

    def get_unit_data(self):
        """Получает данные из Годовой план закупа, листу Поквартально"""
        data = self.google_conn_to_annual_procurement_plan_table.sheet_title.get_all_values()
        # Данные начиная с 4-q строки, заголовки начиная с 3-й
        df = pd.DataFrame(data[3:], columns=data[2])
        return df
    
    # ===
    # Подключение к БД
    # ===
    # def get_funnel_data(self):
    #     """Получает данные из базы данных по запросу supplies_query"""
    #     return Database.read_sql_to_dataframe(funnel_query)


In [16]:
plan = SalesPlan()
df = plan.get_unit_data()

✅ Успешное подключение к Годовой план закупа 2026 -> Поквартально


In [ ]:
# Забираем поквартальный план в шт.
df_quarter = df.copy()[['wild', '1 квартал, шт', '2 квартал, шт', '3 квартал, шт', '4 квартал, шт']]
df_quarter.head()
# Удаляю полные данные по Годовому плану
# del df

,wild,"1 квартал, шт","2 квартал, шт","3 квартал, шт","4 квартал, шт"
0,wild1999,0,0,6000,6000
1,wild2000,0,0,6000,6000
2,wild106,11867,20 000,20 000,15 000
3,wild1944,,,,
4,wild1932,0,0,5 000,5 000
